# The four kinds of machine learning 🔬

Forty-one women had a piece of their breast tumor photographed with a machine
that measures **36 proteins in every single cell** — about 200,000 cells in all.

We'll use that data to meet the **four main kinds of machine learning**. You
don't have to write anything: every cell already works. Run them in order with
`Shift + Enter`, then look for the **🎛️ Try it** boxes and change a number to
see what happens.

|  | predict a **category** | predict a **number** |
|---|---|---|
| **have labels** (supervised) | ① Classification | ② Regression |
| **no labels** (unsupervised) | ③ Clustering | ④ Dimensionality reduction |

In [ ]:
# --- setup: run me first! ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/emcramer/psi-class-2026/main/data/processed"

plt.rcParams.update({"figure.figsize": (8, 5.5), "font.size": 13,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9",
                     "figure.facecolor": "white", "axes.facecolor": "white"})
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
print("ready")

In [ ]:
cells = pd.read_csv(f"{DATA}/cells_sample.csv")
print(cells.shape, "-> 20,000 cells, 16 protein measurements each")
cells.head()

Each **row** is one cell; each **column** is how much of one protein it has —
a fingerprint 16 numbers long. The `celltype` column is the **answer key**:
what a biologist said each cell is. We'll use it, hide it, and predict it.

---
# ① Classification — *supervised, predict a category*

**Supervised** means we *have* the answers and teach the computer to copy them.
We'll show it labeled cells, then test it on cells it has never seen.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

MARKERS = ["CD45", "CD3", "CD8", "CD4", "CD20", "CD68", "CD11c", "MPO",
           "Pan-Keratin", "Beta catenin", "Keratin17", "Vimentin", "SMA",
           "CD31", "FoxP3", "HLA-DR"]

X = StandardScaler().fit_transform(cells[MARKERS])
y = cells["celltype"]

# split: teach on 70% of the cells, test on the other 30%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=0)

model = RandomForestClassifier(n_estimators=120, random_state=0)
model.fit(X_train, y_train)

print(f"accuracy on cells it never saw: {model.score(X_test, y_test):.1%}")

Around **90% correct** on cells it was never taught. Let's *see* the rule it
learned. With just two markers — Keratin (tumor) and CD45 (immune) — we can
draw the decision it makes everywhere on the map.

In [ ]:
from matplotlib.colors import ListedColormap
from sklearn.linear_model import LogisticRegression

def coarse(labels):
    return np.where(labels == "Tumor", "tumor",
            np.where(np.isin(labels, ["Endothelial", "Mesenchymal"]),
                     "other", "immune"))

two = StandardScaler().fit_transform(cells[["Pan-Keratin", "CD45"]])
yc = coarse(cells.celltype.values)
mini = LogisticRegression(max_iter=1000).fit(two, yc)

xx, yy = np.meshgrid(np.linspace(two[:,0].min(), two[:,0].max(), 300),
                     np.linspace(two[:,1].min(), two[:,1].max(), 300))
Z = mini.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
order = {"tumor":0, "immune":1, "other":2}
plt.figure(figsize=(7.5, 6.5))
plt.pcolormesh(xx, yy, np.vectorize(order.get)(Z),
               cmap=ListedColormap(["#cfe0f6","#f9d9c6","#cdeee0"]), shading="auto")
for k, c in [("tumor", BLUE), ("immune", ORANGE), ("other", GREEN)]:
    m = yc == k
    plt.scatter(two[m,0][:1500], two[m,1][:1500], s=6, c=c, alpha=.6, label=k)
plt.xlabel("Keratin  (more = tumor)"); plt.ylabel("CD45  (more = immune)")
plt.legend(markerscale=3); plt.xticks([]); plt.yticks([]); plt.grid(False)
plt.title("The rule the computer drew"); plt.show()

The shaded regions are the rule: land in the blue zone and it calls you a
tumor cell. That is *all a classifier is* — a boundary learned from examples.

> ### 🎛️ Try it
> - In the accuracy cell, swap `RandomForestClassifier(...)` for
>   `LogisticRegression(max_iter=500)`. Simpler model — does it do worse?
> - In the picture, change `["Pan-Keratin", "CD45"]` to `["CD3", "CD8"]`
>   (two T-cell markers). Can those two alone separate the cell types?

---
# ② Regression — *supervised, predict a number*

Same idea — we have answers — but now the answer is a **number**, not a
category. Real question from the paper: a pathologist eyeballs each tumor and
gives it a "TILs" immune score from 1 to 4. **Can the computer's cell count
predict that human score?**

In [ ]:
patients = pd.read_csv(f"{DATA}/patients.csv")
all_cells = pd.read_csv(f"{DATA}/cells_xy.csv.gz")
all_cells["type"] = coarse(all_cells.celltype.values)

# for each patient: what fraction of cells are immune?
immune_frac = all_cells.groupby("SampleID").type.apply(lambda t: (t=="immune").mean())
data = patients.assign(immune_frac=patients.SampleID.map(immune_frac)).dropna(subset=["TIL_score"])

slope, intercept = np.polyfit(data.immune_frac, data.TIL_score, 1)
r2 = np.corrcoef(data.immune_frac, data.TIL_score)[0,1] ** 2

xs = np.linspace(0, data.immune_frac.max(), 50)
plt.figure(figsize=(7.5, 6))
plt.plot(xs, intercept + slope*xs, color=BLUE, lw=2.5, label=f"fitted line (R²={r2:.2f})")
plt.scatter(data.immune_frac, data.TIL_score, s=90, c=ORANGE, edgecolors="white", zorder=3)
plt.xlabel("immune fraction the COMPUTER counted")
plt.ylabel("TIL score a PATHOLOGIST gave")
plt.legend(); plt.title("Predicting a number"); plt.show()

print(f"R² = {r2:.2f}   (1.0 = perfect, 0 = useless)")

The dots trend up the line: more immune cells counted, higher the human score.
The computer's automatic count agrees with the expert's eye about **two-thirds
of the way** (R² = 0.66). That's regression — fit a line, predict a number.

> ### 🎛️ Try it
> - Predict a patient yourself: if the computer counts **40%** immune cells,
>   run `intercept + slope * 0.40`. What TIL score does the line guess?
> - `TIL_score` runs 1–4. What in the data might keep R² from reaching 1.0?

---
# ③ Clustering — *unsupervised, no answer key*

Now we **hide the answers**. Unsupervised learning gets no labels at all — it
just looks for groups that naturally hang together. Watch it rediscover the
cell types on its own.

In [ ]:
from sklearn.cluster import KMeans

labels = KMeans(n_clusters=6, n_init=20, random_state=0).fit_predict(X)
print("the computer sorted 20,000 cells into 6 piles:", np.bincount(labels))

profile = pd.DataFrame(X, columns=MARKERS).groupby(labels).mean()
plt.figure(figsize=(11, 4.5))
plt.imshow(profile, cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="auto")
plt.xticks(range(len(MARKERS)), MARKERS, rotation=45, ha="right")
plt.yticks(range(6), [f"pile {i}" for i in range(6)])
plt.colorbar(label="how much of this protein"); plt.grid(False)
plt.title("Red = lots of it.  Which pile is which cell type?"); plt.show()

### 🔍 Be the biologist

It grouped the cells with **no answer key**. Use the cheat sheet to name each
pile from its red squares, then run the next cell to check.

| marker | cell |
|---|---|
| CD3, CD4, CD8 | T cell |
| CD20 | B cell |
| MPO | neutrophil |
| CD68 | macrophage |
| Pan-Keratin, Keratin17 | tumor |

In [ ]:
answer = pd.crosstab(labels, cells.celltype)
for i in range(6):
    top = profile.loc[i].sort_values(ascending=False).head(3)
    print(f"pile {i}: high in {', '.join(top.index):40s} -> really mostly {answer.loc[i].idxmax()}")

It found T cells, B cells, neutrophils and tumor cells — **without ever being
told they existed.** That's the difference: classification *reproduces* labels
you have; clustering *discovers* groups you don't.

> ### 🎛️ Try it
> Change `n_clusters=6` to `3`, or to `10`. With more piles, do rare cell
> types split off into their own group?

---
# ④ Dimensionality reduction — *unsupervised, simplify*

Each cell is 16 numbers — a point in *16-dimensional* space we can't picture.
**PCA** squashes those 16 numbers down to 2 so we can put every cell on a map.

In [ ]:
from sklearn.decomposition import PCA

xy2 = PCA(n_components=2).fit_transform(X)
plt.figure(figsize=(7.5, 6.5))
for k, c in [("other", GREEN), ("tumor", BLUE), ("immune", ORANGE)]:
    m = coarse(cells.celltype.values) == k
    plt.scatter(xy2[m,0], xy2[m,1], s=4, c=c, alpha=.5, label=k)
plt.legend(markerscale=4); plt.xticks([]); plt.yticks([]); plt.grid(False)
plt.title("16 numbers per cell, squashed onto a 2-D map"); plt.show()

Tumor cells settle in one region, immune cells in another — and the colors
were **painted on afterwards**, never used to build the map. The structure was
already in the numbers. That's the fourth kind: keep the shape of the data
while throwing away dimensions you can't see.

---
# ⑤ Capstone: inventing your own feature 🏆

The four kinds are the toolkit. Real science comes from **feeding them the
right numbers.** Here's the actual finding from this dataset.

First, an honest failure. Does *which* cells a patient has predict survival?

In [ ]:
composition = pd.crosstab(all_cells.SampleID, all_cells.celltype, normalize="index")
composition = composition.loc[composition.index.isin(patients.SampleID)]
groups = KMeans(2, n_init=50, random_state=0).fit_predict(
    StandardScaler().fit_transform(composition))
surv = pd.DataFrame({"SampleID": composition.index, "group": groups}).merge(patients)

def survival_curve(days, died):
    order = np.argsort(days); days, died = np.array(days)[order], np.array(died)[order]
    t, s, alive, n = [0], [1.0], 1.0, len(days)
    for i, d in enumerate(days):
        if died[i]: alive *= 1 - 1/n; t.append(d/365.25); s.append(alive)
        n -= 1
    return t + [days.max()/365.25], s + [alive]

for g, c in [(0, BLUE), (1, ORANGE)]:
    s = surv[surv.group == g]; t, sv = survival_curve(s.survival_days, s.event)
    plt.step(t, sv, where="post", color=c, lw=2.5, label=f"group {g} (n={len(s)})")
plt.ylim(0,1.02); plt.xlabel("years"); plt.ylabel("fraction alive")
plt.title("Grouped by WHICH cells they have"); plt.legend(); plt.show()

The two lines sit on top of each other — cell *composition* tells us almost
nothing about survival. So we invent a better number. Look at two patients with
the **same amount** of immune cells:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, pid in zip(axes, [3, 12]):
    g = all_cells[all_cells.SampleID == pid]
    for k, c in [("other", GREEN), ("tumor", BLUE), ("immune", ORANGE)]:
        s = g[g.type == k]; ax.scatter(s.x, s.y, s=3, c=c, alpha=.75)
    ax.set_title(f"Patient {pid} — {(g.type=='immune').mean():.0%} immune")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_aspect("equal")
plt.show()

Left: immune cells **walled off** in their own territory. Right: **mixed**
right in among the tumor. Same ingredients, different architecture. Let's turn
that into a number — the **mixing score**:

> mixing score = immune cells touching tumor cells ÷ immune cells touching each other

In [ ]:
from scipy.spatial import cKDTree

rows = []
for pid, g in all_cells.groupby("SampleID"):
    pairs = cKDTree(g[["x","y"]].values).query_pairs(30, output_type="ndarray")
    a, b = g.type.values[pairs[:,0]], g.type.values[pairs[:,1]]
    imm_imm = ((a=="immune") & (b=="immune")).sum()
    imm_tum = (((a=="immune") & (b=="tumor")) | ((a=="tumor") & (b=="immune"))).sum()
    rows.append({"SampleID": pid, "mixing": imm_tum/max(imm_imm,1),
                 "n_immune": (g.type=="immune").sum()})
scores = pd.DataFrame(rows)

warm = scores[scores.n_immune >= 250].merge(patients, on="SampleID")
warm["label"] = np.where(warm.mixing < 0.26, "walled off", "mixed")
for lab, c in [("walled off", BLUE), ("mixed", ORANGE)]:
    s = warm[warm.label == lab]; t, sv = survival_curve(s.survival_days, s.event)
    plt.step(t, sv, where="post", color=c, lw=2.5, label=f"{lab} (n={len(s)})")
plt.ylim(0,1.02); plt.xlabel("years"); plt.ylabel("fraction alive")
plt.title("Grouped by WHERE their cells are"); plt.legend(); plt.show()

## That gap is the whole point.

Patients whose immune cells were **mixed into** the tumor did far worse —
about **5× the risk of dying** — than those whose immune cells were **walled
off**. *Which* cells you have: told us nothing. *Where* they are: told us a lot.

You just reproduced a finding from a paper in *Cell*, one of the top journals
in biology, in about ten lines of code.

> ### 🎛️ Try it
> - Change `30` (the touching distance) to `50`. Does the survival gap hold?
> - Change the `0.26` cutoff. How far can you push it before it breaks?